# NLP Project : Text Processing

## 0. Setup

In [1]:
# imports
import re
import random
import numpy as np
import pandas as pd
from pathlib import Path
 
import nltk
from nltk.corpus import stopwords
 
import tensorflow as tf
from keras import Model
import keras.layers as layers
from keras.optimizers import Adam
from keras.regularizers import l2
from keras.layers import TextVectorization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, 
                             accuracy_score, 
                             roc_auc_score, 
                             roc_curve,
                             precision_recall_curve, 
                             average_precision_score)

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import kaleido
kaleido.get_chrome_sync()

nltk.download('stopwords')
nltk.download('punkt')

# initialise random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

I0000 00:00:1774594454.404231     999 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774594456.643059     999 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/davidf_wsl/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/davidf_wsl/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## 1. Loading Dataset

In [2]:
DATA_DIR = Path("../Kaggle_FakeAndRealNewsDataset")
FAKE_NEWS_CSV = DATA_DIR / "Fake.csv"

# validate dataset path
def validate_csv_path(path, label):
    if not path.exists():
        raise FileNotFoundError(f">> [ERROR] {label} not found at: {path.resolve()}\n"
                                f"  → Download from: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset")
    if not path.is_file():
        raise ValueError(f">> [ERROR] {label} exists but is not a file: {path.resolve()}")
    if path.suffix.lower() != ".csv":
        raise ValueError(f">> [ERROR] {label} is not a CSV file: {path.resolve()}")
    
    print(f">> [OK] {label}: {path.resolve()}")
    
validate_csv_path(FAKE_NEWS_CSV, "Fake news dataset")

>> [OK] Fake news dataset: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/TextProcessing/Kaggle_FakeAndRealNewsDataset/Fake.csv


In [3]:
df = pd.read_csv(FAKE_NEWS_CSV)

# Drop rows with missing text or subject
df = df.dropna(subset=["text", "subject"])
df = df[df["text"].str.strip() != ""]

print(f">> Dataset Shape: {df.shape}")
print(f"\n>> --- [Subject Distribution] --- :\n")

display(pd.DataFrame(df['subject'].value_counts()). rename(columns={'subject': 'count'}).reset_index())

>> Dataset Shape: (22851, 4)

>> --- [Subject Distribution] --- :



,subject,count
0,News,9050
1,politics,6433
2,left-news,4309
3,Government News,1498
4,US_News,783
5,Middle-east,778


## 2. Text Preprocessing

In [4]:
# get stop words
stop_words = set(stopwords.words("english"))

# def function to preprocess text - lowercase, remove punctuation, remove stop words
def preprocess(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    
    # join tokens back to string
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(preprocess)

print(">> Preprocessing complete")
display(df.head())

>> Preprocessing complete


,title,text,subject,date,clean_text
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",donald trump wish americans happy new year lea...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",house intelligence committee chairman devin nu...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",friday revealed former milwaukee sheriff david...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",christmas day donald trump announced would bac...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",pope francis used annual christmas day message...


## 3. Label Encoding

In [5]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["subject"])

print(f">> --- [Classes] --- :\n{label_encoder.classes_}")
print("\n>> Label mapping:")

for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i}: {cls}")
 
NUM_CLASSES = len(label_encoder.classes_)
print(f"\n>> Number of classes: {NUM_CLASSES}")

>> --- [Classes] --- :
['Government News' 'Middle-east' 'News' 'US_News' 'left-news' 'politics']

>> Label mapping:
  0: Government News
  1: Middle-east
  2: News
  3: US_News
  4: left-news
  5: politics

>> Number of classes: 6


## 4. Train / Test Split

In [6]:
X_text = df["clean_text"].values
y = df["label"].values

X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text, y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

# print percentage
print(f">> Train set: {len(X_train_text)} samples ({len(X_train_text)/len(X_text)*100:.2f}%)")
print(f">> Validation set: {len(X_val_text)} samples ({len(X_val_text)/len(X_text)*100:.2f}%)")
print(f">> Test set: {len(X_test_text)} samples ({len(X_test_text)/len(X_text)*100:.2f}%)")

>> Train set: 15995 samples (70.00%)
>> Validation set: 3428 samples (15.00%)
>> Test set: 3428 samples (15.00%)


### 4.1. Computing Class-Weights

In [7]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print(">> --- [Class weights] --- :")
for i, (cls, weight) in enumerate(zip(label_encoder.classes_, class_weights)):
    print(f">> {cls:<20} | {weight:.3f}")

>> --- [Class weights] --- :
>> Government News      | 2.544
>> Middle-east          | 4.891
>> News                 | 0.421
>> US_News              | 4.865
>> left-news            | 0.884
>> politics             | 0.592


## 5. TF-IDF Feature Extraction

### 5.1. TF-IDF for ML baselines

In [8]:
MAX_FEATURES = 10_000

tfidf_vectoriser = TfidfVectorizer(max_features=MAX_FEATURES)

X_tfidf_train = tfidf_vectoriser.fit_transform(X_train_text)
X_tfidf_val   = tfidf_vectoriser.transform(X_val_text)
X_tfidf_test  = tfidf_vectoriser.transform(X_test_text)

print(f">> TF-IDF feature matrix shape (train): {X_tfidf_train.shape}")
print(f">> TF-IDF feature matrix shape (val): {X_tfidf_val.shape}")
print(f">> TF-IDF feature matrix shape (test): {X_tfidf_test.shape}")

>> TF-IDF feature matrix shape (train): (15995, 10000)
>> TF-IDF feature matrix shape (val): (3428, 10000)
>> TF-IDF feature matrix shape (test): (3428, 10000)


### 5.2. Integer sequences for CNN-LSTM

In [9]:
VOCAB_SIZE  = [10_000, 101_637, 200_000]    # 101,637
SEQ_LENGTH  = 500        # Sequence Length 500
EMBED_DIM   = 50         # Embedding Dimension 50

seq_splits = []
vectorisers = []

for vocab_size in VOCAB_SIZE:
    # Adapt on the full corpus BEFORE the train/test split so the vocabulary
    # covers all tokens (how the paper fitted on the full dataset).
    vectorise_layer = TextVectorization(
        max_tokens=vocab_size,       # vocabulary cap
        output_mode="int",           # integer indices (same as old texts_to_sequences)
        output_sequence_length=SEQ_LENGTH,  # pads/truncates to fixed length (replaces pad_sequences)
        name=f"text_vectorization_{vocab_size}"  # unique name per vocab size
    )
    vectorise_layer.adapt(X_train_text)
    
    X_seq_train = vectorise_layer(X_train_text).numpy()
    X_seq_val   = vectorise_layer(X_val_text).numpy()
    X_seq_test  = vectorise_layer(X_test_text).numpy()
    
    # Store the splits and vectoriser for later use in training and evaluation
    seq_splits.append((
        X_seq_train, X_seq_val, X_seq_test,
        y_train, y_val, y_test
    ))

    vectorisers.append((vocab_size, vectorise_layer))
    
    print(f">> --- [Vocabulary size: {vocab_size}] --- :")
    print(f">> Train shape: {X_seq_train.shape}")
    print(f">> Val shape:   {X_seq_val.shape}")
    print(f">> Test shape:  {X_seq_test.shape}\n")

>> --- [Vocabulary size: 10000] --- :
>> Train shape: (15995, 500)
>> Val shape:   (3428, 500)
>> Test shape:  (3428, 500)

>> --- [Vocabulary size: 101637] --- :
>> Train shape: (15995, 500)
>> Val shape:   (3428, 500)
>> Test shape:  (3428, 500)

>> --- [Vocabulary size: 200000] --- :
>> Train shape: (15995, 500)
>> Val shape:   (3428, 500)
>> Test shape:  (3428, 500)



## 6. EDA Baselines (Precedent Approaches 1–5)

In [10]:
baseline_models = {
    "Decision Tree  (Prec. 1)": DecisionTreeClassifier(random_state=SEED),
    "Naive Bayes    (Prec. 2)": MultinomialNB(),
    "SVM (Linear)   (Prec. 3)": LinearSVC(random_state=SEED, max_iter=2000),
    "KNN            (Prec. 4)": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Random Forest  (Prec. 5)": RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
}

baseline_results = {}

print(">> --- [Training ML Baselines] --- :")
 
for name, baseline_model in baseline_models.items():
    print(f">> Training {name}...", end=" ", flush=True)
    
    # Train only on the training split
    baseline_model.fit(X_tfidf_train, y_train)

    # Evaluate on the held-out test split
    predictions = baseline_model.predict(X_tfidf_test)
    
    accuracy = accuracy_score(y_test, predictions)
    f1 = classification_report(y_test, 
                               predictions, 
                               target_names=label_encoder.classes_, 
                               zero_division=0, 
                               output_dict=True)["weighted avg"]["f1-score"]
    
    baseline_results[name] = {"model": baseline_model, "predictions": predictions, "accuracy": accuracy, "f1": f1}
    
    print(f"| Accuracy = {accuracy:.3f}, Weighted F1 = {f1:.3f}")

>> --- [Training ML Baselines] --- :
>> Training Decision Tree  (Prec. 1)... | Accuracy = 0.504, Weighted F1 = 0.507
>> Training Naive Bayes    (Prec. 2)... | Accuracy = 0.553, Weighted F1 = 0.491
>> Training SVM (Linear)   (Prec. 3)... | Accuracy = 0.543, Weighted F1 = 0.536
>> Training KNN            (Prec. 4)... | Accuracy = 0.240, Weighted F1 = 0.202
>> Training Random Forest  (Prec. 5)... | Accuracy = 0.532, Weighted F1 = 0.507


## 7. CNN-LSTM Model

### 7.1. Building CNN-LSTM Models

In [11]:
def build_cnn_lstm(vocab_size, seq_length, embed_dim, lstm_units, num_classes, learning_rate, grad_clip, has_attention=False):
    """
    Builds the Fine-Tuned CNN-LSTM model.
 
    Architecture:
        - Embedding → Conv1D → MaxPooling1D → LSTM → Dense → Softmax
        
    Parameters:
        - vocab_size: Size of the vocabulary (input_dim for Embedding).
        - seq_length: Fixed input sequence length (e.g., 500).
        - embed_dim: Dimension of the word embeddings (output_dim for Embedding).
        - lstm_units: Number of hidden units in the LSTM layer.
        - num_classes: Number of output classes for the final Dense layer.
        - learning_rate: Learning rate for the Adam optimizer.
        - grad_clip: Gradient clipping value for the Adam Optimiser.
        - has_attention: Whether to include an attention layer
        
    Note: input_length is omitted from Embedding because TextVectorization
    already guarantees fixed-length sequences (SEQ_LENGTH=500), so Keras
    can infer the shape automatically.
    """
    
    name = f"CNN_LSTM_Model_Vocab{vocab_size}"
    inputs = layers.Input(shape=(seq_length,), name="input_sequence")
    
    # Word embedding layer (Section 4.2.2)
    x = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim, embeddings_regularizer=l2(1e-4), name="embedding")(inputs)
    
    # Convolutional layer (Section 4.2.3) — "6×1 convolutional layer"
    x = layers.Conv1D(filters=64, kernel_size=5, activation="relu", name="conv1d")(x)
    # Max-pooling layer (Section 4.2.4)
    x = layers.MaxPooling1D(pool_size=2, name="maxpool")(x)
    
    if has_attention:
        name = f"CNN_LSTM_Attention_Model_Vocab{vocab_size}"
        # return_sequences=True keeps 3D output (batch, steps, features)
        x = layers.LSTM(units=lstm_units, return_sequences=True, name="lstm")(x)
        # Multi-Head Attention (way forward addition)
        x = layers.MultiHeadAttention(num_heads=2, key_dim=64)(x, x)
        # Add dropout after attention to prevent overfitting on the attention outputs
        x = layers.Dropout(0.3, name="attention_dropout")(x)
        # Pool across the sequence dimension before Dense
        x = layers.GlobalAveragePooling1D()(x)
    else:
        # LSTM layer (Section 4.2.5) — 80 hidden units
        x = layers.LSTM(units=lstm_units, name="lstm")(x)
    
    # Fully connected + Softmax (Section 4.2.6)
    x = layers.Dense(64, activation="relu", name="dense")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="softmax_output")(x)
    
    cnn_lstm_model = Model(inputs, outputs, name=name)
    
    # Attention models are more sensitive to LR — use a lower rate to avoid
    # the loss diverging immediately from epoch 1
    lower_lr = learning_rate * 0.5 if has_attention else learning_rate
    optimiser = Adam(learning_rate=lower_lr, clipvalue=grad_clip) # GradientThreshold = 0.2

    # compile model with sparse_categorical_crossentropy for multi-class classification
    cnn_lstm_model.compile(loss="sparse_categorical_crossentropy", optimizer=optimiser, metrics=["accuracy"])
    return cnn_lstm_model

In [12]:
config = {
    "vocab_size": VOCAB_SIZE,
    "seq_length": SEQ_LENGTH,
    "embed_dim": EMBED_DIM,
    "lstm_units": 80,
    "num_classes": NUM_CLASSES,
    "learning_rate": 0.001,
    "grad_clip": 0.2
}

cnn_lstm_models = []
cnn_lstm_attention_models = []

for vocab_size in config["vocab_size"]:
    print(f"\n>> --- [Building CNN-LSTM model with vocab size: {vocab_size:,}] --- :")
    
    cnn_lstm = build_cnn_lstm(
        vocab_size,
        config["seq_length"], 
        config["embed_dim"], 
        config["lstm_units"], 
        config["num_classes"], 
        config["learning_rate"], 
        config["grad_clip"]
    )
    
    cnn_lstm.summary()
    cnn_lstm_models.append(cnn_lstm)
    
    cnn_lstm_attention = build_cnn_lstm(
        vocab_size, 
        config["seq_length"],
        config["embed_dim"], 
        config["lstm_units"], 
        config["num_classes"], 
        config["learning_rate"], 
        config["grad_clip"],
        has_attention=True
    )

    cnn_lstm_attention.summary()
    cnn_lstm_attention_models.append(cnn_lstm_attention)


>> --- [Building CNN-LSTM model with vocab size: 10,000] --- :


Model: "CNN_LSTM_Model_Vocab10000"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_sequence (InputLayer)     │ (None, 500)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 500, 50)        │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 496, 64)        │        16,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool (MaxPooling1D)          │ (None, 248, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 80)             │        46,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         5,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax_output (Dense)          │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 568,038 (2.17 MB)

 Trainable params: 568,038 (2.17 MB)

...

Model: "CNN_LSTM_Attention_Model_Vocab200000"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 500)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 500, 50)   │ 10,000,000 │ input_sequence[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 496, 64)   │     16,064 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ maxpool             │ (None, 248, 64)   │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 248, 80)   │     46,400 │ maxpool[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 248, 80)   │     41,424 │ lstm[0][0],       │
│ (MultiHeadAttentio… │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_dropout   │ (None, 248, 80)   │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 80)        │          0 │ attention_dropou… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      5,184 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax_output      │ (None, 6)         │        390 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,109,462 (38.56 MB)

 Trainable params: 10,109,462 (38.56 MB)

 Non-trainable params: 0 (0.00 B)

### 7.2. Callbacks

In [13]:
def get_callbacks(use_early_stopping):
    callbacks = [
        # Reduce LR if val_loss plateaus — helps match paper's manual LR tuning
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1)
    ]

    if use_early_stopping:
        # Early stopping to avoid wasted compute
        callbacks.append(EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1))
        
    return callbacks

## 8. Training

In [14]:
EPOCHS = 50
BATCH_SIZE = 16
use_early_stopping = True

model_histories = []
attention_histories = []

def train_models(model_list, attention=False):
    histories = attention_histories if attention else model_histories

    for model_index, (cnn_lstm, split) in enumerate(zip(model_list, seq_splits)):
        X_seq_train, X_seq_val, _, y_train_dl, y_val_dl, _ = split

        vocab_size = config["vocab_size"][model_index]
        
        if attention:
            print(f"\n>> --- [Training CNN-LSTM with Attention and vocab size: {vocab_size:,}] --- :")
        else:
            print(f"\n>> --- [Training CNN-LSTM with vocab size: {vocab_size:,}] --- :")
        
        history = cnn_lstm.fit(
            X_seq_train, y_train_dl,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(X_seq_val, y_val_dl),
            shuffle=True,
            callbacks=get_callbacks(use_early_stopping),
            class_weight=class_weight_dict,
            verbose=1
        )
        
        histories.append(history)

train_models(cnn_lstm_models, attention=False)
train_models(cnn_lstm_attention_models, attention=True)
    
print("\n>> --- [Training Complete] ---")


>> --- [Training CNN-LSTM with vocab size: 10,000] --- :
Epoch 1/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 25s 21ms/step - accuracy: 0.2682 - loss: 1.7437 - val_accuracy: 0.4099 - val_loss: 1.7167 - learning_rate: 0.0010
Epoch 2/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 18s 18ms/step - accuracy: 0.2931 - loss: 1.5580 - val_accuracy: 0.4215 - val_loss: 1.4803 - learning_rate: 0.0010
Epoch 3/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 23s 20ms/step - accuracy: 0.3270 - loss: 1.3929 - val_accuracy: 0.4043 - val_loss: 1.5132 - learning_rate: 0.0010
Epoch 4/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 17s 17ms/step - accuracy: 0.3419 - loss: 1.3926 - val_accuracy: 0.2089 - val_loss: 1.4878 - learning_rate: 0.0010
Epoch 5/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 20s 20ms/step - accuracy: 0.3433 - loss: 1.4043 - val_accuracy: 0.4236 - val_loss: 1.4745 - learning_rate: 0.0010
Epoch 6/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - accuracy: 0.3408 - loss: 1.4082 - val_accuracy: 0.4244 - val_loss: 1.4726 - learning_rate: 0.0010
Epoch 

### 8.1. Saving Models

In [15]:
wORn = "With" if use_early_stopping else "No"
MODELS_OUTPUT_DIR = Path(f"../Out/Models_{wORn}_EarlyStopping")
MODELS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # create if it doesn't exist

def save_model(cnn_lstm_model_lists, has_attention=False):
    for i, cnn_lstm in enumerate(cnn_lstm_model_lists):
        if has_attention:
            model_path = MODELS_OUTPUT_DIR / f"cnn_lstm_attention_model_vocab{config['vocab_size'][i]}.keras"
        else:
            model_path = MODELS_OUTPUT_DIR / f"cnn_lstm_model_vocab{config['vocab_size'][i]}.keras"
        
        # save model to .keras in OUTPUT_DIR
        cnn_lstm.save(model_path)
        if has_attention:
            print(f">> [OK] Saved CNN-LSTM + Attention model with vocab size {config['vocab_size'][i]} to: {model_path.resolve()}")
        else:
            print(f">> [OK] Saved CNN-LSTM model with vocab size {config['vocab_size'][i]} to: {model_path.resolve()}")

save_model(cnn_lstm_models)
save_model(cnn_lstm_attention_models, has_attention=True)

>> [OK] Saved CNN-LSTM model with vocab size 10000 to: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/TextProcessing/Out/Models_With_EarlyStopping/cnn_lstm_model_vocab10000.keras
>> [OK] Saved CNN-LSTM model with vocab size 101637 to: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/TextProcessing/Out/Models_With_EarlyStopping/cnn_lstm_model_vocab101637.keras
>> [OK] Saved CNN-LSTM model with vocab size 200000 to: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/TextProcessing/Out/Models_With_EarlyStopping/cnn_lstm_model_vocab200000.keras
>> [OK] Saved CNN-LSTM + Attention model with vocab size 10000 to: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/TextProcessing/Out/Models_With_EarlyStopping/cnn_lstm_attention_model_vocab10000.keras
>> [OK] Saved CNN-LSTM + Attention model with vocab size 101637 to: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/TextProcessing/Out/Models_With_EarlyStopping/cnn_lstm_attention_model_vocab101637.keras
>> [OK] Saved CNN-LSTM + Attention model with vocab size 200

### 8.2. Training Curves

In [16]:
PLOTS_OUTPUT_DIR = MODELS_OUTPUT_DIR / "Training_Plots"
PLOTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # create if it doesn't exist

for history_index, (history, attention_history) in enumerate(zip(model_histories, attention_histories)):
    for hist, model_type in [(history, "CNN-LSTM"), (attention_history, "CNN-LSTM + Attention")]:
        epochs = list(range(1, len(hist.history["loss"]) + 1))

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=("Loss", "Accuracy")
        )

        # --- Loss curves (left) ---
        fig.add_trace(
            go.Scatter(x=epochs, y=hist.history["loss"],
                    name="Training Loss", line=dict(color="royalblue")),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(x=epochs, y=hist.history["val_loss"],
                    name="Validation Loss", line=dict(color="darkcyan", dash="dash")),
            row=1, col=1
        )

        # --- Accuracy curves (right) ---
        fig.add_trace(
            go.Scatter(x=epochs, y=hist.history["accuracy"],
                    name="Training Accuracy", line=dict(color="darkorange")),
            row=1, col=2
        )
        fig.add_trace(
            go.Scatter(x=epochs, y=hist.history["val_accuracy"],
                    name="Validation Accuracy", line=dict(color="darkred", dash="dash")),
            row=1, col=2
        )

        # Paper target accuracy 0.81
        fig.add_hline(
            y=0.81, row=1, col=2,
            line=dict(color="red", dash="dot", width=1.5),
            annotation_text="Paper target (0.81)",
            annotation_position="top left"
        )

        # --- Layout ---
        fig.update_layout(
            title=dict(
                text=f"{model_type} Training Progress (Loss & Accuracy) - Vocabulary Size: {config['vocab_size'][history_index]:,}",
                x=0.5,
                xanchor="center"
            ),
            width=1100, height=450,
            legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
            hovermode="x unified"
        )

        fig.update_xaxes(title_text="Epoch")
        fig.update_yaxes(title_text="Loss",    row=1, col=1)
        fig.update_yaxes(title_text="Accuracy", row=1, col=2)
        
        fig.show()
        
        # Save plot to file
        if model_type == "CNN-LSTM + Attention":
            model_type_str = "cnn_lstm_attention_model"
        else:
            model_type_str = "cnn_lstm_model"
            
        plot_filename = f"{model_type_str}_vocab{config['vocab_size'][history_index]:,}.png"
        fig.write_image(PLOTS_OUTPUT_DIR / plot_filename)

## 9. Evaluation

In [17]:
# colours for the 6 classes — one per class, consistent across all plots
CLASS_COLORS = px.colors.qualitative.Plotly[:NUM_CLASSES]

def plot_roc_curves(y_true, y_pred_probs, class_names, title):
    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
    
    fig = go.Figure()
    
    # random chance line
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1],
        line=dict(color="black", dash="dash", width=1),
        name="Random chance",
        showlegend=True
    ))
    
    for i, cls in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_score = roc_auc_score(y_true_bin[:, i], y_pred_probs[:, i])
        
        fig.add_trace(go.Scatter(
            x=fpr, y=tpr,
            name=f"{cls} (AUC = {auc_score:.3f})",
            line=dict(color=CLASS_COLORS[i], width=2)
        ))
    
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center"),
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate",
        width=750, height=550,
        legend=dict(x=0.98, y=0.02, xanchor="right", yanchor="bottom"),
        hovermode="x unified"
    )
    
    fig.update_xaxes(range=[0, 1])
    fig.update_yaxes(range=[0, 1])
    
    return fig


def plot_pr_curves(y_true, y_pred_probs, class_names, title):
    """Precision-Recall curve per class — AP score in legend."""
    
    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
    
    fig = go.Figure()
    
    for i, cls in enumerate(class_names):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        ap_score = average_precision_score(y_true_bin[:, i], y_pred_probs[:, i])
        
        fig.add_trace(go.Scatter(
            x=recall, y=precision,
            name=f"{cls} (AP = {ap_score:.3f})",
            line=dict(color=CLASS_COLORS[i], width=2)
        ))
    
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center"),
        xaxis_title="Recall",
        yaxis_title="Precision",
        width=750, height=550,
        legend=dict(x=0.98, y=0.98, xanchor="right", yanchor="top"),
        hovermode="x unified"
    )
    
    fig.update_xaxes(range=[0, 1])
    fig.update_yaxes(range=[0, 1])
    
    return fig

In [18]:
EVAL_PLOTS_OUTPUT_DIR = MODELS_OUTPUT_DIR / "Eval_Plots"
EVAL_PLOTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # create if it doesn't exist

def evaluate_model(model_list, sequence_splits, has_attention=False):
    # --- Run for each vocab size ---
    for model_index, (cnn_lstm, split) in enumerate(zip(model_list, sequence_splits)):
        _, _, X_seq_test, _, _, y_test_dl = split

        vocab_label = f"Vocab {config['vocab_size'][model_index]:,}"
        
        # get predicted probabilities
        y_pred_probs = cnn_lstm.predict(X_seq_test, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        
        # print classification report
        class_report = classification_report(
            y_test_dl, 
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0,
            digits=3
        )
        print(f">> Classification Report ({vocab_label}):")
        print(class_report)
        
        # --- ROC-AUC ---
        if has_attention:
            model_type_str = "CNN-LSTM + Attention"
        else:
            model_type_str = "CNN-LSTM"

        fig_roc = plot_roc_curves(
            y_test_dl, 
            y_pred_probs,
            label_encoder.classes_,
            title=f"{model_type_str} ({vocab_label}) - ROC Curve"
        )
        fig_roc.show()
        # save ROC plot to file
        fig_roc.write_image(EVAL_PLOTS_OUTPUT_DIR / f"{cnn_lstm.name.lower()}_ROC.png")

        # --- PR-AP ---
        fig_pr = plot_pr_curves(
            y_test_dl, y_pred_probs,
            label_encoder.classes_,
            title=f"{model_type_str} ({vocab_label}) - PR Curve"
        )
        
        fig_pr.show()
        # save PR plot to file
        fig_pr.write_image(EVAL_PLOTS_OUTPUT_DIR / f"{cnn_lstm.name.lower()}_PR.png")

print("\n>> --- [Evaluating CNN-LSTM Models] --- :")
evaluate_model(cnn_lstm_models, seq_splits)
print("\n>> --- [Evaluating CNN-LSTM + Attention Models] --- :")
evaluate_model(cnn_lstm_attention_models, seq_splits, has_attention=True)


>> --- [Evaluating CNN-LSTM Models] --- :
>> Classification Report (Vocab 10,000):
                 precision    recall  f1-score   support

Government News      0.119     0.876     0.209       225
    Middle-east      0.136     0.052     0.075       116
           News      0.889     0.951     0.919      1358
        US_News      0.410     0.607     0.490       117
      left-news      0.296     0.037     0.066       647
       politics      0.250     0.004     0.008       965

       accuracy                          0.465      3428
      macro avg      0.350     0.421     0.294      3428
   weighted avg      0.505     0.465     0.412      3428



>> Classification Report (Vocab 101,637):
                 precision    recall  f1-score   support

Government News      0.119     0.920     0.211       225
    Middle-east      0.000     0.000     0.000       116
           News      0.998     0.979     0.988      1358
        US_News      0.487     0.940     0.641       117
      left-news      0.246     0.025     0.045       647
       politics      0.393     0.025     0.047       965

       accuracy                          0.492      3428
      macro avg      0.374     0.481     0.322      3428
   weighted avg      0.577     0.492     0.449      3428



>> Classification Report (Vocab 200,000):
                 precision    recall  f1-score   support

Government News      0.000     0.000     0.000       225
    Middle-east      0.185     0.043     0.070       116
           News      0.947     0.909     0.927      1358
        US_News      0.452     0.803     0.578       117
      left-news      0.330     0.957     0.491       647
       politics      0.000     0.000     0.000       965

       accuracy                          0.569      3428
      macro avg      0.319     0.452     0.345      3428
   weighted avg      0.459     0.569     0.482      3428




>> --- [Evaluating CNN-LSTM + Attention Models] --- :
>> Classification Report (Vocab 10,000):
                 precision    recall  f1-score   support

Government News      0.191     0.391     0.257       225
    Middle-east      0.667     0.017     0.034       116
           News      0.987     0.977     0.982      1358
        US_News      0.493     0.966     0.653       117
      left-news      0.368     0.784     0.500       647
       politics      0.615     0.008     0.016       965

       accuracy                          0.597      3428
      macro avg      0.554     0.524     0.407      3428
   weighted avg      0.686     0.597     0.528      3428



>> Classification Report (Vocab 101,637):
                 precision    recall  f1-score   support

Government News      0.137     0.196     0.161       225
    Middle-east      0.500     0.026     0.049       116
           News      0.994     0.973     0.984      1358
        US_News      0.496     0.966     0.655       117
      left-news      0.354     0.768     0.485       647
       politics      0.514     0.075     0.130       965

       accuracy                          0.598      3428
      macro avg      0.499     0.501     0.411      3428
   weighted avg      0.648     0.598     0.552      3428



>> Classification Report (Vocab 200,000):
                 precision    recall  f1-score   support

Government News      0.163     0.347     0.222       225
    Middle-east      0.491     0.448     0.468       116
           News      0.978     0.976     0.977      1358
        US_News      0.488     0.521     0.504       117
      left-news      0.355     0.748     0.482       647
       politics      0.000     0.000     0.000       965

       accuracy                          0.584      3428
      macro avg      0.412     0.507     0.442      3428
   weighted avg      0.498     0.584     0.526      3428



### 9.1. Inference Testing

In [19]:
def run_inference_on_samples(model_list, seq_splits):
    X_test_text_raw = X_test_text

    all_sample_dfs = []
    
    for (cnn_lstm_model, split) in zip(model_list, seq_splits):
        print(f">> Running Inference on {cnn_lstm_model.name.lower()}")
        _, _, X_seq_test_raw, _, _, y_test_sample = split

        y_pred_probs = cnn_lstm_model.predict(X_seq_test_raw, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)

        correct_idx = np.where(y_pred == y_test_sample)[0][:3]
        incorrect_idx = np.where(y_pred != y_test_sample)[0][:3]
        sample_idx = np.concatenate([correct_idx, incorrect_idx])

        sample_df = pd.DataFrame({
            "model": cnn_lstm_model.name.lower(),
            "true_label": label_encoder.inverse_transform(y_test_sample[sample_idx]),
            "pred_label": label_encoder.inverse_transform(y_pred[sample_idx]),
            "confidence": np.max(y_pred_probs[sample_idx], axis=1),
            "correct": y_pred[sample_idx] == y_test_sample[sample_idx],
            "text": X_test_text_raw[sample_idx],
            "cleaned_text": [preprocess(t) for t in X_test_text_raw[sample_idx]]
        })
        
        all_sample_dfs.append(sample_df)

    return pd.concat(all_sample_dfs, ignore_index=True)

In [20]:
cnn_lstm_sample_results = run_inference_on_samples(cnn_lstm_models, seq_splits)
attention_sample_results = run_inference_on_samples(cnn_lstm_attention_models, seq_splits)

# combine into one and save to CSV
combined_sample_results = pd.concat([cnn_lstm_sample_results, attention_sample_results],ignore_index=True)

output_file = MODELS_OUTPUT_DIR / f"all_inference_samples_{'with_early_stopping' if use_early_stopping else 'no_early_stopping'}.csv"
combined_sample_results.to_csv(output_file, index=False)

print(f"\n>> [OK] Saved inference samples to: {output_file.resolve()}")

>> Running Inference on cnn_lstm_model_vocab10000
>> Running Inference on cnn_lstm_model_vocab101637
>> Running Inference on cnn_lstm_model_vocab200000
>> Running Inference on cnn_lstm_attention_model_vocab10000
>> Running Inference on cnn_lstm_attention_model_vocab101637
>> Running Inference on cnn_lstm_attention_model_vocab200000

>> [OK] Saved inference samples to: /home/davidf_wsl/ARI5121 - AppliedNLP_Project/TextProcessing/Out/Models_With_EarlyStopping/all_inference_samples_with_early_stopping.csv
